# THRIFTY: AI-Powered Clothing Condition Grading & Category Classification
### Integrated PBL Semester 5 — Information Technology
**Team Lead & Lead Developer:** Tiara Febrianie  
**ML Engineer / System Architect:** Joseph Atem Deng Aruei  
**Lecturer (Machine Learning):** Muhammad Afif Hendrawan, S.Kom., M.T.  
**Lecturer (Mobile Programming):** Habibie Ed Dien, S.Kom., M.T.  

---
## 1. Project Background & Objectives
Thrifty is a location-based secondhand fashion marketplace mobile application. 
One of the primary challenges in secondhand fashion is the **inconsistency of clothing condition descriptions** between sellers. 
To solve this, Thrifty introduces an integrated **AI Condition Grading** and **Category Classification** model:
- **Target Categories (4):** Shirt, T-Shirt, Pants, Jacket
- **Target Condition Grades (4):**
  - `Grade A`: Like New (little to no visible sign of use)
  - `Grade B`: Very Good (minor visible signs of use)
  - `Grade C`: Good / Usable (visible signs of use or minor wear)
  - `Grade D`: Significant Wear (noticeable defects or substantial wear)
- **Backbone Architecture:** MobileNetV2 (Transfer Learning from ImageNet)
- **Success Criteria:** Minimum 80% test accuracy on clothing condition classification.

## 2. Environment Setup and Imports

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version:      {keras.__version__}")

## 3. Dataset Inspection & Preprocessing
The dataset is structured in `ml/data/processed/` across `train`, `val`, and `test` splits (80/10/10 ratio).

In [ ]:
CATEGORIES = ['Shirt', 'T-Shirt', 'Pants', 'Jacket']
GRADES = ['Grade A', 'Grade B', 'Grade C', 'Grade D']
IMG_SIZE = (224, 224)

data_dir = os.path.join('..', 'data', 'processed')
train_dir = os.path.join(data_dir, 'train')
print("Classes in train:", os.listdir(train_dir)[:8])

## 4. MobileNetV2 Dual-Head Transfer Learning Architecture
A shared MobileNetV2 backbone extracts spatial and texture features. Two distinct dense heads perform multi-task classification:
1. `category_output`: Softmax output over 4 clothing types
2. `grade_output`: Softmax output over 4 condition grades

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = layers.Input(shape=(224, 224, 3), name='clothing_image_input')
x = layers.Rescaling(scale=2.0, offset=-1.0)(inputs)
features = base_model(x, training=False)
pooled = layers.GlobalAveragePooling2D(name='global_pooling')(features)
shared_dense = layers.Dense(128, activation='relu', name='shared_features')(pooled)
shared_dense = layers.BatchNormalization()(shared_dense)
shared_dense = layers.Dropout(0.3)(shared_dense)

# Category Head
cat_dense = layers.Dense(64, activation='relu')(shared_dense)
cat_output = layers.Dense(len(CATEGORIES), activation='softmax', name='category_output')(cat_dense)

# Condition Grade Head
grade_dense = layers.Dense(64, activation='relu')(shared_dense)
grade_output = layers.Dense(len(GRADES), activation='softmax', name='grade_output')(grade_dense)

model = models.Model(inputs=inputs, outputs=[cat_output, grade_output], name='Thrifty_MobileNetV2_MultiTask')
model.summary()

## 5. Model Evaluation and Confusion Matrices

In [ ]:
# Load evaluation summary
reports_path = os.path.join('..', 'reports', 'evaluation_report.json')
if os.path.exists(reports_path):
    with open(reports_path, 'r') as f:
        eval_report = json.load(f)
    print(f"Category Classification Test Accuracy: {eval_report['metrics']['category_accuracy']*100:.2f}%")
    print(f"Clothing Condition Grading Test Accuracy: {eval_report['metrics']['grade_accuracy']*100:.2f}%")
else:
    print("Run train.py first to generate the evaluation report.")

## 6. Real-Time Inference Demo (End-to-End Simulation)

In [ ]:
def test_inference(image_path):
    from service.app import preprocess_image, heuristic_visual_analysis
    with open(image_path, 'rb') as f:
        img_bytes = f.read()
    tensor, pil_img = preprocess_image(img_bytes)
    cat, cat_scores, grade, grade_scores = heuristic_visual_analysis(pil_img)
    print(f"Predicted Category: {cat} (Confidence: {cat_scores[cat]*100:.1f}%)")
    print(f"Predicted Grade:    {grade} (Confidence: {grade_scores[grade]*100:.1f}%)")
    return pil_img

print("Inference function ready for testing on uploaded clothing images.")